# ODIR-5K Ablation Study — CNN (EXP 1, 2, 3)

**3 thực nghiệm CNN (EfficientNet-B0):**
- EXP 1: Ảnh gốc, không tiền xử lý
- EXP 2: Tiền xử lý (ROI+Ben Graham+CLAHE), không MixUp/CutMix
- EXP 3: Tiền xử lý + MixUp + CutMix

In [ ]:
# ── CELL 1: Cài thư viện ──────────────────────────────────
!pip install timm albumentations pyyaml scikit-learn -q
print('✅ Libraries installed')

In [ ]:
# ── CELL 2: Kiểm tra GPU ──────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  KHÔNG CÓ GPU! Vào Settings → Accelerator → GPU T4')

In [ ]:
# ── CELL 3: Thiết lập đường dẫn + Giải nén code ──────────
import os, sys, zipfile, shutil

# Dataset odir5k-code mount tại đây
CODE_INPUT = '/kaggle/input/odir5k-code'

# Giải nén src.zip, configs.zip, splits_clean.zip vào /kaggle/working/code/
WORK_CODE = '/kaggle/working/code'
os.makedirs(WORK_CODE, exist_ok=True)

for zname in ['src.zip', 'configs.zip', 'splits_clean.zip']:
    zpath = f'{CODE_INPUT}/{zname}'
    if os.path.exists(zpath):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK_CODE)
        print(f'✅ Giải nén: {zname}')
    else:
        print(f'❌ Không tìm thấy: {zpath}')

# Copy file .py
for fname in ['train.py', 'evaluate.py']:
    src = f'{CODE_INPUT}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, WORK_CODE)
        print(f'✅ Copy: {fname}')

# Thêm vào Python path để import src.*
sys.path.insert(0, WORK_CODE)

# Đường dẫn dùng trong notebook
CODE_DIR   = WORK_CODE
SPLITS_DIR = f'{WORK_CODE}/splits_clean'

# Ảnh raw ODIR-5K từ Kaggle public dataset
RAW_DIR = '/kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training Images'

# Working dirs cho ảnh đã xử lý và kết quả
ROI_DIR = '/kaggle/working/preprocessed_images'
ENH_DIR = '/kaggle/working/enhanced_images'
RES_DIR = '/kaggle/working/results'

for d in [ROI_DIR, ENH_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'\nCODE_DIR   : {CODE_DIR}')
print(f'SPLITS_DIR : {SPLITS_DIR}')
print(f'RAW_DIR    : {RAW_DIR}')
print('\nKiểm tra files:')
for f in os.listdir(WORK_CODE):
    print(f'  {f}')


In [ ]:
# ── CELL 4: Kiểm tra đường dẫn thực tế ───────────────────
# Nếu RAW_DIR sai, cell này sẽ báo lỗi → sửa Cell 3
from pathlib import Path

raw_path = Path(RAW_DIR)
if not raw_path.exists():
    print('❌ RAW_DIR không tồn tại! Kiểm tra cấu trúc:')
    for root, dirs, files in os.walk('/kaggle/input/ocular-disease-recognition-odir5k'):
        depth = root.replace('/kaggle/input/ocular-disease-recognition-odir5k','').count('/')
        print('  ' + '  '*depth + os.path.basename(root) + '/')
        if depth >= 4: break
else:
    n_raw = len(list(raw_path.glob('*.jpg')))
    print(f'✅ RAW_DIR OK — {n_raw} ảnh')

splits_path = Path(SPLITS_DIR)
if splits_path.exists():
    print(f'✅ SPLITS_DIR OK — {list(splits_path.glob("*.csv"))}')
else:
    print(f'❌ SPLITS_DIR không tồn tại: {SPLITS_DIR}')

# Kiểm tra import src
try:
    from src.dataset import ODIRDataset
    from src.models import build_model
    from src.loss import MultiTaskLoss
    from src.mixup import MixUpCollator
    from src.cutmix import CutMixCollator
    print('✅ src.* import OK!')
except Exception as e:
    print(f'❌ Import lỗi: {e}')

In [ ]:
# ── CELL 5: ROI Crop ──────────────────────────────────────
import cv2, numpy as np
from pathlib import Path
from tqdm import tqdm

def crop_roi(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    rows, cols = np.where(mask)
    if len(rows) == 0: return cv2.resize(img, (512,512))
    r0,r1,c0,c1 = rows.min(),rows.max(),cols.min(),cols.max()
    return cv2.resize(img[r0:r1+1, c0:c1+1], (512,512))

imgs = list(Path(RAW_DIR).glob('*.jpg'))
print(f'Bắt đầu ROI Crop {len(imgs)} ảnh...')
for p in tqdm(imgs, desc='ROI Crop'):
    img = cv2.imread(str(p))
    if img is None: continue
    cv2.imwrite(str(Path(ROI_DIR)/p.name), crop_roi(img))
print(f'✅ ROI Crop xong: {len(list(Path(ROI_DIR).glob("*.jpg")))} ảnh')

In [ ]:
# ── CELL 6: Ben Graham + CLAHE ────────────────────────────
def ben_graham(img, sigma_ratio=1/6, scale=128):
    h,w = img.shape[:2]
    sigma = int(max(h,w)*sigma_ratio)
    if sigma%2==0: sigma+=1
    local = cv2.GaussianBlur(img.astype(np.float32),(0,0),sigmaX=sigma)
    return np.clip(img.astype(np.float32)-local+scale,0,255).astype(np.uint8)

def apply_clahe(img):
    lab = cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l),a,b]),cv2.COLOR_LAB2BGR)

roi_imgs = list(Path(ROI_DIR).glob('*.jpg'))
print(f'Ben Graham + CLAHE trên {len(roi_imgs)} ảnh...')
for p in tqdm(roi_imgs, desc='Enhance'):
    img = cv2.imread(str(p))
    if img is None: continue
    cv2.imwrite(str(Path(ENH_DIR)/p.name), apply_clahe(ben_graham(img)))
print(f'✅ Enhanced xong: {len(list(Path(ENH_DIR).glob("*.jpg")))} ảnh')

In [ ]:
# ── CELL 7: Load metadata & pos_weight ───────────────────
import json
meta     = json.load(open(f'{SPLITS_DIR}/metadata.json'))
age_mean = meta['age_stats']['mean']
age_std  = meta['age_stats']['std']
LABELS   = meta['labels']
pos_weight = torch.FloatTensor([meta['class_weights'][l] for l in LABELS])

print(f'Labels: {LABELS}')
print(f'age_mean={age_mean:.2f}, age_std={age_std:.2f}')
print('pos_weight:', {l:round(v,2) for l,v in zip(LABELS,pos_weight.tolist())})

In [ ]:
# ── CELL 8: Hàm training đầy đủ ──────────────────────────
import yaml, time
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, roc_auc_score
from src.dataset import ODIRDataset
from src.transforms import get_transforms
from src.models import build_model
from src.loss import MultiTaskLoss
from src.mixup import MixUpCollator
from src.cutmix import CutMixCollator

def run_experiment(config_path, img_dir_override=None):
    """Chạy 1 thực nghiệm từ file YAML config."""
    cfg = yaml.safe_load(open(config_path))
    exp_name  = cfg['experiment_name']
    model_type = cfg.get('model_type', 'cnn')
    tr_cfg    = cfg['training']
    aug_cfg   = cfg.get('augmentation', {})
    img_size  = tr_cfg['img_size']
    batch     = tr_cfg['batch_size']
    epochs    = tr_cfg['epochs']
    patience  = tr_cfg.get('early_stopping_patience', 5)

    # Override img_dir nếu cần (Kaggle đường dẫn khác local)
    if img_dir_override:
        img_dir = img_dir_override
    elif 'enhanced' in cfg.get('img_dir',''):
        img_dir = ENH_DIR
    else:
        img_dir = RAW_DIR

    out_dir = f'{RES_DIR}/{exp_name}'
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*55}\n  {exp_name}\n  img_dir={img_dir}\n{'='*55}")

    tf_train = get_transforms('train', img_size)
    tf_val   = get_transforms('val',   img_size)

    def mk_ds(split, tf):
        return ODIRDataset(f'{SPLITS_DIR}/{split}.csv', img_dir, tf, age_mean, age_std)

    ds_train = mk_ds('train', tf_train)
    ds_val   = mk_ds('val',   tf_val)
    ds_test  = mk_ds('test',  tf_val)

    # Train loader — MixUp/CutMix nếu bật
    use_mixup  = aug_cfg.get('use_mixup', False)
    use_cutmix = aug_cfg.get('use_cutmix', False)
    base_kw = dict(batch_size=batch, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    if use_mixup and use_cutmix:
        import random
        class _MixCut:
            def __init__(self):
                self.mx = MixUpCollator(alpha=aug_cfg.get('mixup_alpha',0.4), prob=1.0)
                self.cx = CutMixCollator(alpha=aug_cfg.get('cutmix_alpha',1.0), prob=1.0)
            def __call__(self, b): return self.mx(b) if random.random()<0.5 else self.cx(b)
        train_loader = DataLoader(ds_train, **base_kw, collate_fn=_MixCut())
        print('[Aug] MixUp + CutMix xen kẽ')
    elif use_mixup:
        train_loader = DataLoader(ds_train, **base_kw, collate_fn=MixUpCollator(alpha=0.4, prob=0.5))
        print('[Aug] MixUp only')
    else:
        train_loader = DataLoader(ds_train, **base_kw)
        print('[Aug] Không dùng MixUp/CutMix')

    val_loader  = DataLoader(ds_val,  batch_size=batch, num_workers=2)
    test_loader = DataLoader(ds_test, batch_size=batch, num_workers=2)

    model = build_model(model_type=model_type, pretrained=True,
                        img_size=img_size, variant=cfg.get('model',{}).get('variant','tiny')).to(device)
    pw = pos_weight.to(device)
    criterion = MultiTaskLoss(pos_weight=pw, lam=cfg['loss'].get('lam',0.1))
    optimizer = torch.optim.AdamW(model.parameters(),
                                   lr=cfg['optimizer'].get('lr',3e-4),
                                   weight_decay=cfg['optimizer'].get('weight_decay',0.01))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    def run_epoch(loader, mode):
        model.train() if mode=='train' else model.eval()
        tot=0; probs=[]; tgts=[]; ap=[]; at=[]
        ctx = torch.enable_grad() if mode=='train' else torch.no_grad()
        with ctx:
            for batch in loader:
                imgs=batch['image'].to(device); lbl=batch['labels'].to(device); age=batch['age'].to(device)
                out=model(imgs); loss,_=criterion(out['logits'],lbl,out['age_pred'],age)
                if mode=='train':
                    optimizer.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
                tot+=loss.item()*imgs.size(0)
                probs.extend(torch.sigmoid(out['logits']).detach().cpu().tolist())
                tgts.extend(lbl.detach().cpu().tolist())
                ap.extend(out['age_pred'].squeeze(1).detach().cpu().tolist())
                at.extend(age.squeeze(1).detach().cpu().tolist())
        import numpy as np
        p=np.array(probs); t=np.array(tgts)
        f1=f1_score(t,(p>=0.5).astype(int),average='macro',zero_division=0)
        try: auc=roc_auc_score(t,p,average='macro')
        except: auc=-1.0
        ap2=(np.array(ap)*age_std+age_mean); at2=(np.array(at)*age_std+age_mean)
        mae=float(np.mean(np.abs(ap2-at2)))
        return {'loss':tot/len(loader.dataset),'f1':f1,'auc':auc,'mae':mae}

    best_f1=0; wait=0; best_state=None; log=[]
    for ep in range(1,epochs+1):
        t0=time.time()
        tm=run_epoch(train_loader,'train'); vm=run_epoch(val_loader,'val')
        scheduler.step()
        print(f'Ep {ep:02d}/{epochs} | Train={tm["loss"]:.4f} | Val F1={vm["f1"]:.4f} AUC={vm["auc"]:.4f} MAE={vm["mae"]:.1f}y [{time.time()-t0:.1f}s]')
        log.append({'ep':ep,'train_loss':tm['loss'],'val_f1':vm['f1'],'val_auc':vm['auc'],'val_mae':vm['mae']})
        if vm['f1']>best_f1:
            best_f1=vm['f1']; wait=0
            best_state=model.state_dict()
            torch.save(best_state, f'{out_dir}/best.pth')
        else:
            wait+=1
            if wait>=patience: print(f'  ⏹ Early stop at ep {ep}'); break

    model.load_state_dict(best_state)
    test_m=run_epoch(test_loader,'test')
    print(f'\nTEST: F1={test_m["f1"]:.4f} AUC={test_m["auc"]:.4f} MAE={test_m["mae"]:.1f}y')
    result={'exp':exp_name,'best_val_f1':best_f1,'test':test_m,'log':log}
    json.dump(result, open(f'{out_dir}/results.json','w'), indent=2)
    return result

print('✅ Training function ready!')

In [ ]:
# ── CELL 9: EXP 1 — Ảnh gốc, không tiền xử lý ───────────
r1 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_1_cnn_no_preprocess.yaml',
    img_dir_override = RAW_DIR,   # Dùng ảnh gốc
)

In [ ]:
# ── CELL 10: EXP 2 — Tiền xử lý, không MixUp/CutMix ──────
r2 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_2_cnn_preprocess_no_aug.yaml',
    img_dir_override = ENH_DIR,   # Dùng ảnh enhanced
)

In [ ]:
# ── CELL 11: EXP 3 — Tiền xử lý + MixUp + CutMix ─────────
r3 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_3_cnn_preprocess_with_aug.yaml',
    img_dir_override = ENH_DIR,   # Dùng ảnh enhanced
)

In [ ]:
# ── CELL 12: Bảng so sánh kết quả ────────────────────────
import pandas as pd

exp_labels = {
    'exp_1_cnn_no_preprocess':      'EXP 1: Raw (không xử lý)',
    'exp_2_cnn_preprocess_no_aug':  'EXP 2: Enhanced (không aug)',
    'exp_3_cnn_preprocess_with_aug':'EXP 3: Enhanced + MixUp+CutMix',
}

rows = []
for r in [r1,r2,r3]:
    t = r['test']
    rows.append({
        'Thực nghiệm': exp_labels.get(r['exp'], r['exp']),
        'Best Val F1': round(r['best_val_f1'],4),
        'Test F1-macro': round(t['f1'],4),
        'Test AUC-ROC': round(t['auc'],4),
        'Test Age MAE': round(t['mae'],2),
    })

df = pd.DataFrame(rows).set_index('Thực nghiệm')
print(df.to_string())

# Phân tích delta
d12 = r2['test']['f1'] - r1['test']['f1']
d23 = r3['test']['f1'] - r2['test']['f1']
d13 = r3['test']['f1'] - r1['test']['f1']
print(f'\n[PHÂN TÍCH]')
print(f'Tiền xử lý (EXP2-EXP1):    {d12:+.4f} → {"✅ Có ích" if d12>0.01 else "≈ Không đáng kể"}')
print(f'MixUp+CutMix (EXP3-EXP2):  {d23:+.4f} → {"✅ Có ích" if d23>0.005 else "≈ Không đáng kể"}')
print(f'Tổng cải thiện (EXP3-EXP1): {d13:+.4f}')

# Lưu bảng markdown
md_lines = ['| Thực nghiệm | Test F1 | Test AUC | Test MAE |','|---|---|---|---|']
for _, row in df.iterrows():
    md_lines.append(f"| {_.strip()} | {row['Test F1-macro']} | {row['Test AUC-ROC']} | {row['Test Age MAE']} |")
open(f'{RES_DIR}/comparison_table.md','w').write('\n'.join(md_lines))
print(f'\n✅ Bảng so sánh lưu tại: {RES_DIR}/comparison_table.md')

In [ ]:
# ── CELL 13: Vẽ Learning Curves ──────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2,figsize=(14,5))
colors = ['#e74c3c','#3498db','#2ecc71']
exp_names = ['EXP1: Raw','EXP2: Enhanced','EXP3: Enhanced+Aug']

for r,c,lbl in zip([r1,r2,r3], colors, exp_names):
    eps = [x['ep'] for x in r['log']]
    f1s = [x['val_f1'] for x in r['log']]
    maes = [x['val_mae'] for x in r['log']]
    axes[0].plot(eps,f1s,c=c,label=lbl,linewidth=2)
    axes[1].plot(eps,maes,c=c,label=lbl,linewidth=2)

axes[0].set(title='Val F1-macro theo Epoch',xlabel='Epoch',ylabel='F1-macro')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Val Age MAE theo Epoch',xlabel='Epoch',ylabel='MAE (years)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RES_DIR}/cnn_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Learning curves lưu tại {RES_DIR}/cnn_learning_curves.png')